<a href="https://colab.research.google.com/github/ochilovu2010/IOAI/blob/main/NitroAI/Polyglot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#https://judge.nitro-ai.org/competitions/roai-2025/lot-2-2026/1/view

In [73]:
!unzip "/content/train_data.zip" -d "/content/train/"

Archive:  /content/train_data.zip
replace /content/train/embeddings.npy? [y]es, [n]o, [A]ll, [N]one, [r]ename: N


In [74]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
import torch.nn.functional as F
import torch

master_embeddings = np.load("/content/train/embeddings.npy")
shuffled_subtask1 = np.load("/content/train/subtask1.npy")
shuffled_subtask2 = np.load("/content/train/subtask2.npy")

In [75]:
def find_mapping_iterative(master_embeddings, shuffled_subtask, initial_k_indices, num_iterations=5, confident_fraction=0.1):
    known_map = {idx: idx for idx in initial_k_indices}

    final_mapped_cols = None

    for iteration in range(num_iterations):
        current_reference_indices_master = np.array(list(known_map.keys()))
        current_reference_indices_shuffled = np.array([known_map[k] for k in current_reference_indices_master])

        master_known_ref_embeddings = master_embeddings[current_reference_indices_master]
        shuffled_known_ref_embeddings = shuffled_subtask[current_reference_indices_shuffled]

        projected_master = master_embeddings @ master_known_ref_embeddings.T
        projected_shuffled = shuffled_subtask @ shuffled_known_ref_embeddings.T

        distances = torch.cdist(torch.from_numpy(projected_master), torch.from_numpy(projected_shuffled))

        master_indices_full, mapped_cols_full = linear_sum_assignment(distances)

        final_mapped_cols = mapped_cols_full

        if iteration < num_iterations - 1:
            assigned_distances = distances[master_indices_full, mapped_cols_full]

            sorted_indices = np.argsort(assigned_distances)

            num_to_select = max(len(initial_k_indices), int(len(master_indices_full) * confident_fraction))

            top_confident_master_indices = master_indices_full[sorted_indices[:num_to_select]]
            top_confident_shuffled_indices = mapped_cols_full[sorted_indices[:num_to_select]]

            known_map = {idx: idx for idx in initial_k_indices}
            for i in range(len(top_confident_master_indices)):
                master_idx = top_confident_master_indices[i]
                shuffled_idx = top_confident_shuffled_indices[i]
                known_map[master_idx] = shuffled_idx

    return final_mapped_cols

In [76]:
K1_initial_anchors = 400
initial_k1_indices = list(range(K1_initial_anchors))
master_columns = find_mapping_iterative(master_embeddings, shuffled_subtask1, initial_k1_indices)

In [77]:
K2_initial_anchors = 20
initial_k2_indices = list(range(K2_initial_anchors))
new_columns = find_mapping_iterative(master_embeddings, shuffled_subtask2, initial_k2_indices)

In [78]:
new_columns.shape

(4000,)

In [79]:
submission = pd.DataFrame({
  "subtaskID": 4000*[1]+4000*[2],
  "datapointID": list(range(4000))+list(range(4000)),
  "answer": np.concatenate((master_columns,new_columns))
})

In [80]:
submission.to_csv('submission.csv', index = False)